<a href="https://colab.research.google.com/github/tmvenkadesh/AgenticAI/blob/main/tool_agent_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tool Agent with LangGraph & Ollama

This notebook demonstrates a LangGraph-based agent that routes queries to different tools (technical vs general) using an Ollama LLM with Langfuse observability.

## 1. Install Dependencies

In [6]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

downloading uv 0.11.1 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [7]:
!uv pip install langchain-ollama langgraph langfuse pydantic python-dotenv langgraph-checkpoint-sqlite  --system


Using Python 3.12.13 environment at: /usr
Resolved 55 packages in 727ms
Prepared 2 packages in 17ms
Installed 2 packages in 3ms
 + langgraph-checkpoint-sqlite==3.0.3
 + sqlite-vec==0.1.7


## 2. Ins**tall ollama**

In [8]:
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 6 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (3,863 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [12]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [13]:
import subprocess
subprocess.Popen(["ollama", "serve"])
import time
time.sleep(3)  # wait for server to start


In [22]:
!ollama pull llama3.2

## 3. Set Environment Variables

Replace the placeholder values below with your actual Langfuse keys.

In [14]:
import os
from google.colab import userdata

# Set environment variables

os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_PUBLIC_KEY"] =  userdata.get("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_BASE_URL"] = "https://cloud.langfuse.com"

## 4. Imports

In [15]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from typing import Annotated, TypedDict
from langfuse.langchain import CallbackHandler
import operator
from langfuse import get_client
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

## 5. Setup Langfuse & Config

In [16]:
langfuse = get_client()
langfuse_handler = CallbackHandler()

config = {
    "callbacks": [langfuse_handler],
    "configurable": {"thread_id": "1"}
}

## 6. Define Agent State & Memory

In [17]:
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]


conn = sqlite3.connect("sqlite_memory.sqlite", check_same_thread=False)
checkpoint_memory = SqliteSaver(conn=conn)

## 7. Initialize LLM

In [18]:
llm = ChatOllama(model="llama3.2", temperature=0)

## 8. Define Tools

In [19]:
@tool
def handle_technical_query(question: str) -> str:
    """Handle technical, programming, coding, software engineering,
    or computer science related questions. Use this tool when the user
    asks about writing code, scripts, debugging, algorithms, frameworks,
    APIs, databases, JSON parsing, or any technology-related topic."""
    print("--Handling Technical query via tool call--")
    return f"Here is some python code for: {question}"


@tool
def handle_general_query(question: str) -> str:
    """Handle general knowledge questions like geography, history, science,
    culture, or everyday topics. Use this tool ONLY when the question is NOT
    about programming or technology."""
    print("--Handling General query via tool call--")
    return f"Here is your general question's answer for: {question}"


_tools = [handle_technical_query, handle_general_query]

## 9. Create Agent & Build Graph

In [20]:
# this is complied node
_agent = create_agent(
    model=llm,
    tools=_tools,
    system_prompt="""You are a helpful assistant. Your goal is to answer the user's question accurately.
You have two tools at your disposal:
1. `handle_technical_query`: Use this for any question related to programming, code, algorithms, or software engineering.
2. `handle_general_query`: Use this for general knowledge questions about topics like history, science, or geography.

First, analyze the user's message.
- If the question is technical, you MUST call the `handle_technical_query` tool.
- If the question is non-technical, you MUST call the `handle_general_query` tool.
- If the user is just making conversation (e.g., "hello"), respond naturally without using a tool.

After a tool returns its output, formulate a final, user-friendly answer based on the tool's result."""
)


'''def tools_condition(state: AgentState) -> str:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "__end__"


def Call_model(state: AgentState):
    """Primary function that calls the LLM Agent"""
    messages = state["messages"]
    response = _agent.invoke(
        {"messages": messages},
        config=config
    )
    return {"messages": response["messages"]}


tool_node = ToolNode(_tools)

workflow = StateGraph(AgentState)
workflow.add_node("agent", Call_model)
workflow.add_node("tools", tool_node)

workflow.add_conditional_edges(
    "agent",
    tools_condition,
    {
        "tools": "tools",
        "__end__": "__end__"
    }
)
workflow.add_edge('tools', 'agent')

workflow.set_entry_point("agent")

app = workflow.compile(checkpointer=checkpoint_memory)'''

'def tools_condition(state: AgentState) -> str:\n    last_message = state["messages"][-1]\n    if last_message.tool_calls:\n        return "tools"\n    return "__end__"\n\n\ndef Call_model(state: AgentState):\n    """Primary function that calls the LLM Agent"""\n    messages = state["messages"]\n    response = _agent.invoke(\n        {"messages": messages},\n        config=config\n    )\n    return {"messages": response["messages"]}\n\n\ntool_node = ToolNode(_tools)\n\nworkflow = StateGraph(AgentState)\nworkflow.add_node("agent", Call_model)\nworkflow.add_node("tools", tool_node)\n\nworkflow.add_conditional_edges(\n    "agent",\n    tools_condition,\n    {\n        "tools": "tools",\n        "__end__": "__end__"\n    }\n)\nworkflow.add_edge(\'tools\', \'agent\')\n\nworkflow.set_entry_point("agent")\n\napp = workflow.compile(checkpointer=checkpoint_memory)'

## 10. Test - Technical Question

In [23]:
print("=== Technical Question ===")

result = _agent.invoke(
    {"messages": [HumanMessage(content="write some python script to parse json")]},
    config=config
)
print("Tool used:", result["messages"][-1].content)

=== Technical Question ===
Tool used: I apologize for the mistake in my previous response. Here's a Python script to parse JSON:

```python
import json

def parse_json(json_string):
    """
    Parse a JSON string into a Python dictionary.

    Args:
        json_string (str): The JSON string to be parsed.

    Returns:
        dict: A Python dictionary representing the JSON data.
    """
    try:
        # Attempt to parse the JSON string
        return json.loads(json_string)
    except json.JSONDecodeError as e:
        # Handle any errors that occur during parsing
        print(f"Error parsing JSON: {e}")
        return None

# Example usage:
json_string = '{"name": "John", "age": 30, "city": "New York"}'
parsed_data = parse_json(json_string)

if parsed_data is not None:
    # Print the parsed data as a Python dictionary
    print(parsed_data)
```

This script defines a function `parse_json` that takes a JSON string as input and attempts to parse it into a Python dictionary using t

## 11. Test - General Question

In [ ]:
print("=== General Question ===")

result = _agent.invoke(
    {"messages": [HumanMessage(content="capital of india")]},
    config=config
)
print("Tool used:", result["messages"][-1].content)

## 12. Try Your Own Query

In [24]:
user_query = input("Provide your query ")

result = _agent.invoke(
    {"messages": [HumanMessage(content=user_query)]},
    config=config
)
print("Response:", result["messages"][-1].content)

Hello how are you hello how are you
Response: Hello! I'm doing well, thank you for asking! How can I assist you today?
